# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
# ============================================================
# STEP 1 — CLONE REPOSITORY + LOAD DATASET
# ============================================================

# Clone the repository
!git clone https://github.com/muhammadfaseehkhattak/FlyRank-ML-Internship.git

# Move into the repository
%cd /content/FlyRank-ML-Internship

# Import pandas for working with the dataset
import pandas as pd
from pathlib import Path

# Define the dataset location
DATA_PATH = Path(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Basic verification
print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Number of columns:", len(df.columns))

Cloning into 'FlyRank-ML-Internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 138 (delta 50), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.88 MiB | 6.38 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/FlyRank-ML-Internship
Dataset loaded successfully.
Shape: (30000, 44)
Number of columns: 44


In [7]:
# ============================================================
# SECTION 1 — RANKED ACTIONS + REASON CODES
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

DATA_PATH = Path(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


# ------------------------------------------------------------
# 2. Create target
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# ------------------------------------------------------------
# 3. Create log traffic features
# ------------------------------------------------------------

for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]:
    df[f"log_{col}"] = np.log1p(
        pd.to_numeric(df[col], errors="coerce").fillna(0)
    )


# ------------------------------------------------------------
# 4. Same 27 features used in Week 5/6
# ------------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

MODEL_FEATURES = (
    MODEL_NUMERIC_FEATURES
    + MODEL_CATEGORICAL_FEATURES
)


# ------------------------------------------------------------
# 5. Create the same client-grouped split
# ------------------------------------------------------------

X = df[MODEL_FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()


# ------------------------------------------------------------
# 6. Verify client separation
# ------------------------------------------------------------

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print()
print("SPLIT CHECK")
print("=" * 50)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients))
)


# ------------------------------------------------------------
# 7. Same Logistic Regression pipeline
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            MODEL_NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            MODEL_CATEGORICAL_FEATURES
        ),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)


# ------------------------------------------------------------
# 8. Train on training clients only
# ------------------------------------------------------------

logistic_model.fit(
    X_train,
    y_train
)

print()
print("Logistic Regression trained successfully.")


# ------------------------------------------------------------
# 9. Score unseen test-client pages
# ------------------------------------------------------------

test_scores = logistic_model.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# 10. Build ranked action queue
# ------------------------------------------------------------

action_queue = df.iloc[test_idx].copy()

action_queue["decline_risk_score"] = test_scores

action_queue = action_queue.sort_values(
    "decline_risk_score",
    ascending=False
).reset_index(drop=True)

action_queue["priority_rank"] = (
    np.arange(len(action_queue)) + 1
)


# ------------------------------------------------------------
# 11. Create human-readable reason codes
# ------------------------------------------------------------

def assign_reason(row):

    reasons = []

    if row["decline_risk_score"] >= 0.75:
        reasons.append("HIGH_DECLINE_RISK")

    if row["days_since_last_update"] >= 91:
        reasons.append("STALE_CONTENT")

    if row["ctr"] < 0.50:
        reasons.append("LOW_CTR")

    if row["avg_position"] > 10:
        reasons.append("LOW_SEARCH_POSITION")

    if row["content_age_days"] >= 270:
        reasons.append("OLDER_CONTENT")

    if row["engagement_rate"] < 50:
        reasons.append("LOW_ENGAGEMENT")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return " | ".join(reasons)


action_queue["reason_code"] = action_queue.apply(
    assign_reason,
    axis=1
)


# ------------------------------------------------------------
# 12. Recommended action
# ------------------------------------------------------------

def assign_action(row):

    if (
        row["days_since_last_update"] >= 91
        and row["decline_risk_score"] >= 0.75
    ):
        return "REVIEW_FOR_REFRESH"

    if row["decline_risk_score"] >= 0.75:
        return "HIGH_PRIORITY_REVIEW"

    return "REVIEW_IF_CAPACITY_ALLOWS"


action_queue["recommended_action"] = action_queue.apply(
    assign_action,
    axis=1
)


# ------------------------------------------------------------
# 13. Display top 20
# ------------------------------------------------------------

print()
print("RANKED ACTION QUEUE — TOP 20")
print("=" * 80)

print(
    action_queue[
        [
            "priority_rank",
            "content_id",
            "decline_risk_score",
            "recommended_action",
            "reason_code",
            "days_since_last_update",
            "content_age_days",
            "ctr",
            "avg_position"
        ]
    ].head(20).to_string(index=False)
)


# ------------------------------------------------------------
# 14. Validation
# ------------------------------------------------------------

print()
print("QUEUE CHECK")
print("=" * 50)

print("Queue size:", len(action_queue))
print(
    "Top-50 declining pages:",
    int(
        action_queue.head(50)["is_declining_label"].sum()
    ),
    "out of 50"
)

Dataset loaded successfully.
Shape: (30000, 44)

SPLIT CHECK
Training clients: 25
Testing clients: 7
Client overlap: 0

Logistic Regression trained successfully.

RANKED ACTION QUEUE — TOP 20
 priority_rank           content_id  decline_risk_score   recommended_action                                                                                        reason_code  days_since_last_update  content_age_days  ctr  avg_position
             1 content_a928cb66d230            0.957357 HIGH_PRIORITY_REVIEW                                                       HIGH_DECLINE_RISK | LOW_CTR | LOW_ENGAGEMENT                      20               117 0.00           4.2
             2 content_7be5f150dc65            0.954591 HIGH_PRIORITY_REVIEW                                                       HIGH_DECLINE_RISK | LOW_CTR | LOW_ENGAGEMENT                      20                96 0.00           5.9
             3 content_c82bc0c24241            0.946278 HIGH_PRIORITY_REVIEW                     

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: The ranked queue is intended to help SEO and content teams prioritize pages for human review. It highlights pages with higher model-estimated decline risk and provides simple reason codes to support review.

Limits: The model is a decision-support tool, not an automatic decision-maker. A high decline-risk score does not prove that a page will decline, and the reason codes do not establish causation. Results are based on this dataset and the current validation setup, so they should be re-validated on new clients or future data before production use.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review**

Before taking action on a ranked page, a content or SEO specialist should review the page's actual content, search intent, recent updates, CTR, search position, engagement metrics, and overall business context. The model should be used to prioritize review, while the final decision remains with a human who can assess factors the model may not capture.

**No-go list**

The system should never automatically:

Delete or permanently remove content.

*   Rewrite or publish content without human review.
*   Change important SEO strategy based only on the model score.
*   Assume that high decline risk means the page will definitely decline.
*   Treat the reason codes as proof of causation.
*   Make irreversible changes without human approval.












## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored over time to identify when the model may no longer reflect current content performance. Key triggers include:

Precision@50 declines consistently on newly evaluated data.

Feature distributions change, such as major shifts in CTR, search position, or content age.

The decline rate changes substantially compared with the training data.

New clients, content types, or patterns appear that were not represented during training.

Human reviewers find the ranked queue less useful for prioritizing pages.


If these signals persist, the model should be re-evaluated using recent data and retrained if necessary. Retraining should only happen after checking the data quality, feature definitions, and validation design.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/FlyRank-ML-Internship/work/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = OUTPUT_DIR / "week7_ranked_action_queue.csv"

action_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Queue exported successfully.")
print("Rows:", len(action_queue))
print("Saved to:", OUTPUT_PATH)

Queue exported successfully.
Rows: 6163
Saved to: /content/FlyRank-ML-Internship/work/outputs/week7_ranked_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.